# Gu Ecosystem Simulator 手动添加修改手册

**版本**：基于当前架构（三层属性 + CombatContext + EffectResult 管道 + 触发时机）

本手册帮助你逐步、安全地扩展系统：
- 添加新**特质** (Trait)
- 添加新**技能** (Skill)
- 添加新**性格** (Personality)

我们优先使用**纯数值或基于已有属性**的效果（通过 EffectResult 字段如 `damageAdd`、`fleeChanceBonus`、`defenseRateBonus` 等），避免立即污染核心伤害公式。你可以后续再引入新类型。

## 架构速览（重要）

- **类型**：`src/core/types.ts`
- **属性计算**：`src/core/stats.ts`（性格修正 + 衍生值）
- **效果定义**：
  - 特质 → `src/core/traits.ts`（`TRAIT_DEFINITIONS` + `getTraitEffects`）
  - 技能 → `src/core/skills.ts`（`SKILL_DEFINITIONS` + `tryActivateSkill`）
- **效果应用**：`src/core/combat.ts`（每回合调用触发器并应用 EffectResult）
- **蛊生命周期**：`src/core/gu.ts`（`createRandomGu`、`acquireTrait`、`tryLevelUp`）
- **模拟主循环**：`src/core/engine.ts`
- **UI**：`src/App.vue` + `src/components/BattleView.vue`
- **常量**：`src/utils/constants.ts`

扩展核心原则：
1. 新效果通过 `EffectResult` 返回（不直接改公式）。
2. 性格统一在 `getPersonalityModifiers` 管理。
3. 获得/进化走 `acquireTrait`（自动支持 level + acquisitions）。
4. 每回合效果在 `resolveRound` → `applyEffectsToContext` 中生效。
5. 记得更新 UI 显示和文档。

## 1. 添加新特质（Trait）

### 要定义什么
- `id`：唯一英文 key
- `name`：中文显示名
- `type`：`offense` / `defense` / `utility` / `mutation`
- `description`：说明
- 效果：返回 `EffectResult` 对象（使用已有字段如 `damageAdd`、`fleeChanceBonus` 等）
- （可选）进化加成：用 `trait.level` 做线性或对数提升

### 必须修改的文件
1. `src/core/types.ts`（如果需要新衍生字段）
2. `src/core/traits.ts`（定义 + 效果逻辑）
3. `src/core/stats.ts`（如果影响新属性或性格）
4. `src/core/combat.ts`（可选：特殊触发报告）
5. `src/core/gu.ts`（可选：初始生成概率）
6. UI 文件（显示）
7. `src/utils/constants.ts`（数值可调）
8. `ARCHITECTURE.md`（文档）

### 示例：添加「毒鳞」特质（On-Hit 附加毒伤）

**定义**：
```ts
{
  id: 'poison_scales',
  name: '毒鳞',
  type: 'offense',
  stackable: false,
  description: '被攻击时对攻击者造成毒伤。',
}
```

**修改位置与代码**：

**文件：`src/core/traits.ts`**（最重要）

在 `TRAIT_DEFINITIONS` 数组末尾添加：

In [ ]:
{
  id: 'poison_scales',
  name: '毒鳞',
  type: 'offense',
  stackable: false,
  description: '被攻击时对攻击者造成毒伤。',
}

在 `applyTraitTrigger` 函数的 switch 里添加 case（放在 unstable 后面）：

In [ ]:
case 'poison_scales':
  if (trigger === 'on_hit') {
    const dot = 2 + (lvl - 1) * 0.8;  // 线性成长，lvl 来自 trait.level
    defender.hp = Math.max(1, defender.hp - dot);
    return { 
      log: `因为毒鳞Lv.${lvl}，所以蛊#${defender.id}对攻击者造成${dot}点毒伤` 
    };
  }
  break;

**文件：`src/core/gu.ts`**（让新蛊有概率初始获得）

在 `createRandomGu` 的 initialTraits 循环后（或 if 随机）：

In [ ]:
if (Math.random() < 0.25) {
  acquireTrait(gu, { 
    id: 'poison_scales', 
    name: '毒鳞', 
    type: 'offense' 
  } as any);
}

**文件：`src/core/stats.ts`**（如果想让性格影响这个特质效果）

在 `getDerivedStats` 中为新效果准备字段（如果需要），并在 `getPersonalityModifiers` 里给某些性格加 `poisonBonus` 等（可选，先用已有字段）。

**文件：`src/App.vue` 和 `src/components/BattleView.vue`**（UI 显示）

确保 traits 列表使用：
```ts
{{ t.name }} Lv.{{ t.level || 1 }}
```
（我们之前已经统一了这个格式，新特质会自动显示）。

**文件：`src/utils/constants.ts`**（推荐）

把毒伤基础值抽出来：
```ts
POISON_DOT_BASE: 2,
POISON_DOT_PER_LEVEL: 0.8,
```
然后在 traits.ts 里引用 `FOOD` 风格的常量（需 import）。

**文件：`ARCHITECTURE.md`**（文档）

在「特质系统」章节补充新特质的触发时机和效果描述。

### 生效验证
1. 重启 `pnpm tauri dev`
2. 创建或升级蛊直到获得「毒鳞」
3. 进入 1v1 战斗，观察 BattleView 日志是否出现「因为毒鳞Lv.X，所以...」
4. 检查 HP 是否有额外扣除

## 2. 添加新技能（Skill）

### 要定义什么
- `id`、`name`、`description`
- `damageType`: 'physical' | 'special'
- `mpCost`: number
- `baseActivationChance`: number（会被 effectiveSkillUsageRate + luck 修正）
- 效果：返回 `EffectResult[]`（支持临时 `damageMult`、`fleeChanceBonus` 等）

### 必须修改的文件
1. `src/core/types.ts`（扩展 SkillDefinition 如果需要）
2. `src/core/skills.ts`（定义 + 发动逻辑）
3. `src/core/combat.ts`（通常已有调用，可增强时机）
4. `src/core/gu.ts`（可选：初始带技能）
5. UI（日志已自动显示）
6. constants + ARCHITECTURE.md

### 示例：添加「狂暴冲撞」技能

**定义**：
```ts
{
  id: 'berserk_charge',
  name: '狂暴冲撞',
  description: '本回合物理攻击大幅提升，消耗 MP。',
  damageType: 'physical',
  mpCost: 7,
  baseActivationChance: 0.30,
}
```

**修改位置与代码**：

**文件：`src/core/skills.ts`**

In [ ]:
// 在 SKILL_DEFINITIONS 数组末尾添加
{
  id: 'berserk_charge',
  name: '狂暴冲撞',
  description: '本回合物理攻击大幅提升，消耗 MP。',
  damageType: 'physical',
  mpCost: 7,
  baseActivationChance: 0.30,
}

在 `tryActivateSkill` 里增加：

In [ ]:
if (skill.id === 'berserk_charge') {
  effects.push({
    damageMult: 1.6,
    log: `因为狂暴冲撞，蛊#${gu.id}本回合攻击力大幅提升！`
  });
}

// 返回 SkillResult 时会自动把 effects 应用

**文件：`src/core/combat.ts`**（增强）

在 `resolveRound` 技能尝试后（如果需要根据技能改 damageType）：

In [ ]:
if (skillResult?.activated && skillResult.skill.damageType) {
  ctx.damageType = skillResult.skill.damageType;
}
// 然后 applyEffectsToContext(skillResult.effects, ctx)

**文件：`src/core/gu.ts`**（可选）

在 `createRandomGu` 里给某些性格或随机蛊预设技能（当前技能是战斗中动态尝试，不是常驻列表，可按需扩展 `gu.skills` 字段）。

**文件：`src/App.vue` + `BattleView.vue`**：技能日志会通过 `roundLogs` 自动出现在战斗日志，无需额外 UI（除非要做技能选择界面）。

**文件：`src/utils/constants.ts`**：
```ts
BERSERK_CHARGE_MP_COST: 7,
BERSERK_CHARGE_BASE_CHANCE: 0.30,
```

**文件：`ARCHITECTURE.md`**：补充技能扩展说明。

### 生效验证
- 战斗中观察是否偶尔触发「因为狂暴冲撞...」
- 检查 MP 是否扣除（在 tryActivateSkill 返回后处理）
- 伤害是否明显提升（通过 damageMult 走 calculateDamage）

## 3. 添加新性格（Personality）

### 要定义什么
- 枚举值（在 `Personality` 类型中）
- 对各衍生属性的修正（乘数/加成）
- （可选）对 Meta 属性（luck、mutationRate、skillUsageRate）的影响

### 必须修改的文件
1. `src/core/types.ts`（添加枚举值）
2. `src/core/stats.ts`（最重要：getPersonalityModifiers + getMetaStats）
3. `src/core/gu.ts`（初始属性 bias + 移动影响）
4. `src/core/combat.ts` / `engine.ts`（通常不需要，效果已自动流入）
5. UI（显示性格描述）
6. 文档

### 示例：添加「天真」性格

**定义**：天真 → 增加幸运、略微提高技能触发、降低逃跑倾向。

**修改位置与代码**：

**文件：`src/core/types.ts`**

In [ ]:
export type Personality = 
  | 'aggressive' 
  | 'cautious' 
  | 'opportunistic' 
  | 'balanced'
  | 'naive';   // 新增

**文件：`src/core/stats.ts`**（核心）

在 `getPersonalityModifiers` 里添加 case：

In [ ]:
case 'naive': // 天真
  return {
    atkMult: 0.95,
    defMult: 1.05,
    spdMult: 0.95,
    fleeChanceBonus: -0.08,
    skillUsageRateBonus: 0.08,
    critChanceBonus: 0.05,
  };

在 `getMetaStats` 里补充影响：

In [ ]:
const p = gu.personality;
if (p === 'naive') {
  luck += 8;
  mutationRate += 0.03;
  // skillUsageRate 已在 personality modifiers 里处理
}

**文件：`src/core/gu.ts`**（初始差异）

在 `createRandomGu` 里：

In [ ]:
if (personality === 'naive') {
  // 天真蛊初始 MP 稍高、攻击稍低（示例）
  // 可在此直接调整 base stats
}

**文件：`src/App.vue` + `BattleView.vue`**：在性格显示处增加描述文字，例如：
```ts
if (personality === 'naive') desc = '天真：幸运高，技能较易触发，但较少逃跑';
```

**文件：`ARCHITECTURE.md`**：更新性格影响表格。

### 生效验证
- 创建不同性格的蛊，观察初始属性差异
- 战斗中检查 fleeChance、critChance、effectiveSkillUsageRate 是否受性格影响（可临时在 console 或 inspector 打印）
- 逃跑概率应符合性格设定（胆小更高，天真更低）

## 4. 通用检查清单（每次扩展后执行）

- [ ] `types.ts`：新字段/枚举已添加？
- [ ] `stats.ts`：性格修正和衍生计算已覆盖？
- [ ] 效果是否只通过 `EffectResult` 返回？
- [ ] `combat.ts` / `gu.ts`：在正确时机被调用？
- [ ] `createRandomGu` / `tryLevelUp` / `acquireTrait`：新内容能被获得？
- [ ] UI：至少在 BattleView 或选中面板能看到？
- [ ] `constants.ts`：数值已抽离，便于调优？
- [ ] `ARCHITECTURE.md`：扩展点已更新？
- [ ] 重启 `pnpm tauri dev` 并测试多场战斗

## 5. 常见扩展场景速查

- **想让效果支持进化**：直接用 `trait.level`，`acquireTrait` 已自动处理 3 次 → 升 1 级。
- **想加新触发时机**（如 `on_flee`）：
  1. `types.ts` 加到 `TraitTrigger`
  2. 在调用方（combat/gu/engine）对应位置调用 `getTraitEffects('on_flee', ctx)`
  3. 在 traits.ts / skills.ts 注册处理器
- **想让性格影响移动以外**：全部走 `getPersonalityModifiers`，然后在需要的地方取用。
- **想加持续效果**（dot、buff）：在 `engine.ts` 的 `tick` 里定期调用 `getTraitEffects('passive', fakeContext)` 并应用。
- **想完全不污染公式**：只使用现有 EffectResult 字段 + 性格 bonus。所有公式只读 `getDerivedStats` 返回的值。

## 6. 测试建议

1. 创建/升级蛊直到获得新内容
2. 进入 1v1 战斗（自动或逐步）
3. 观察 BattleView 日志是否出现预期文案
4. 检查 HP、MP、逃跑概率等数值是否符合预期
5. 多跑几场，验证随机性（尤其是逃跑和技能触发）

祝扩展愉快！如果需要针对某个具体新内容（比如你想加的某个特质）生成更精确的 diff 代码，随时告诉我。